# Cyberbullying Classification

**NLP Text Classification Project** — Detect hate speech / cyberbullying in tweets.

Models: TF-IDF + MultinomialNB (baseline) · TF-IDF + Logistic Regression · TF-IDF + SVM
Dataset: Cardiff NLP tweet_eval (hate split) via HuggingFace
Target: `label` (0 = not hate, 1 = hate)

## 2 · Project Overview

We build classifiers to detect **cyberbullying / hate speech** in tweets. This is a binary NLP classification task using the `cardiffnlp/tweet_eval` hate-speech split from HuggingFace datasets. We use traditional NLP approaches (TF-IDF features) with multiple classifiers, as well as demonstrate the baseline approach.

Note: For production NLP classification, a fine-tuned ModernBERT or XLM-RoBERTa would be preferred. We keep this notebook focused on classical ML approaches for educational purposes.

## 3 · Learning Objectives

By the end of this notebook you will be able to:
1. Load text classification datasets from HuggingFace.
2. Perform EDA on text data (text length distribution, class balance).
3. Build TF-IDF feature representations from text.
4. Train and compare multiple text classifiers (MultinomialNB, LogisticRegression, SVM).
5. Evaluate text classification with accuracy, precision, recall, F1, and confusion matrix.
6. Understand the challenges of hate speech detection.

## 4 · Problem Statement

Given a tweet's text, classify whether it contains **hate speech / cyberbullying** (label=1) or not (label=0). This has real-world applications in content moderation, online safety, and social media monitoring.

## 5 · Why This Project Matters

- **Content moderation** at scale requires automated detection of harmful text.
- Hate speech detection is one of the most important socially-impactful NLP tasks.
- Understanding classifier tradeoffs (precision vs recall) is critical — a missed hate-speech message has different cost than a false alarm.
- Text classification fundamentals transfer to many other NLP tasks.

## 6 · Dataset Overview

| Property | Value |
|----------|-------|
| **Source** | HuggingFace `cardiffnlp/tweet_eval` (hate split) |
| **Split** | train |
| **Samples** | ~9,000 (capped to 8,000) |
| **Features** | `text` (tweet text) |
| **Target** | `label` (0 = not hate, 1 = hate) |
| **Task** | Binary text classification |

## 7 · Dataset Source and License Notes

- **Source**: [CardiffNLP tweet_eval](https://huggingface.co/datasets/cardiffnlp/tweet_eval) on HuggingFace.
- **License**: Research use. The dataset collects public tweets for NLP benchmarking.
- **Limitations**: Crowdsourced annotations may not perfectly reflect all definitions of hate speech. Tweet context (images, threads) is not available.

## 8 · Environment Setup

In [ ]:
import subprocess, sys

def _install(pkg, imp=None):
    try: __import__(imp or pkg)
    except ImportError: subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

for p, i in [("datasets",None),("scikit-learn","sklearn")]:
    _install(p, i)
print("All packages ready.")

## 9 · Imports

In [ ]:
import warnings, os, json
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                             f1_score, confusion_matrix, classification_report,
                             ConfusionMatrixDisplay)

SAVE_DIR = os.path.dirname(os.path.abspath("__file__"))
SEED = 42
np.random.seed(SEED)
print("Core imports loaded.")

## 10 · Configuration / Constants

In [ ]:
MAX_SAMPLES = 8000
TFIDF_MAX_FEATURES = 10000
TEST_SIZE = 0.2
print(f"Max samples: {MAX_SAMPLES}, TF-IDF features: {TFIDF_MAX_FEATURES}")

## 11 · Dataset Download or Loading

We load the hate-speech split from HuggingFace's `tweet_eval` dataset.

In [ ]:
from datasets import load_dataset

ds = load_dataset("cardiffnlp/tweet_eval", "hate", split="train")
df = ds.to_pandas()

# Standardize columns
df = df[["text", "label"]].dropna()
if len(df) > MAX_SAMPLES:
    df = df.sample(n=MAX_SAMPLES, random_state=SEED).reset_index(drop=True)

print(f"Dataset shape: {df.shape}")
print(df.head())

## 12 · Data Validation Checks

In [ ]:
print(f"Missing values:\n{df.isnull().sum()}")
print(f"\nDuplicate rows: {df.duplicated().sum()}")
print(f"\nLabel distribution:\n{df['label'].value_counts()}")
print(f"\nLabel proportions:\n{df['label'].value_counts(normalize=True)}")
assert df["text"].notna().all(), "Text column has nulls"
assert df["label"].notna().all(), "Label column has nulls"
print("\nAll validation checks passed.")

## 13 · Exploratory Data Analysis

We examine text length distribution and label balance.

In [ ]:
df["text_len"] = df["text"].str.len()
df["word_count"] = df["text"].str.split().str.len()

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

df["text_len"].hist(bins=50, ax=axes[0], color="steelblue", edgecolor="white")
axes[0].set_title("Text Length Distribution")
axes[0].set_xlabel("Characters")

df["word_count"].hist(bins=50, ax=axes[1], color="coral", edgecolor="white")
axes[1].set_title("Word Count Distribution")
axes[1].set_xlabel("Words")

df["label"].value_counts().plot.bar(ax=axes[2], color=["steelblue", "coral"])
axes[2].set_title("Label Distribution")
axes[2].set_xticklabels(["Not Hate (0)", "Hate (1)"], rotation=0)

plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "eda_text_stats.png"), dpi=100)
plt.show()

print(f"\nText length stats:\n{df['text_len'].describe()}")
print(f"\nWord count stats:\n{df['word_count'].describe()}")

## 14 · Target Analysis

In [ ]:
label_counts = df["label"].value_counts()
print("Class distribution:")
print(label_counts)
imbalance_ratio = label_counts.min() / label_counts.max()
print(f"\nImbalance ratio: {imbalance_ratio:.3f}")
if imbalance_ratio < 0.5:
    print("Dataset is moderately imbalanced — will use class_weight='balanced'.")
else:
    print("Dataset is relatively balanced.")

## 15 · Train/Validation/Test Split Strategy

We use a stratified 80/20 train/test split to preserve class distribution.

In [ ]:
X = df["text"]
y = df["label"]

X_train_text, X_test_text, y_train, y_test = train_test_split(
    X, y, test_size=TEST_SIZE, random_state=SEED, stratify=y
)
print(f"Train: {len(X_train_text)}, Test: {len(X_test_text)}")
print(f"Train label distribution:\n{y_train.value_counts(normalize=True)}")

## 16 · Preprocessing Strategy

For text classification, we use **TF-IDF vectorization** to convert text into numerical features. TF-IDF captures word importance relative to the corpus, which works well as a baseline for text classification.

In [ ]:
vectorizer = TfidfVectorizer(max_features=TFIDF_MAX_FEATURES, ngram_range=(1, 2),
                             stop_words="english", min_df=2)
X_train = vectorizer.fit_transform(X_train_text)
X_test = vectorizer.transform(X_test_text)

print(f"TF-IDF matrix — Train: {X_train.shape}, Test: {X_test.shape}")

## 17 · Feature Engineering

TF-IDF with bigrams serves as our feature engineering. The vectorizer extracts unigram and bigram features, removes English stop words, and filters out very rare terms (min_df=2).

In [ ]:
feature_names = vectorizer.get_feature_names_out()
print(f"Total TF-IDF features: {len(feature_names)}")
print(f"Sample features: {list(feature_names[:20])}")

## 18 · Baseline Model

We start with **MultinomialNB** — the classic baseline for text classification.

In [ ]:
baseline = MultinomialNB(alpha=1.0)
baseline.fit(X_train, y_train)
baseline_pred = baseline.predict(X_test)

print("Baseline — MultinomialNB:")
print(f"  Accuracy : {accuracy_score(y_test, baseline_pred):.4f}")
print(f"  Precision: {precision_score(y_test, baseline_pred, average='weighted'):.4f}")
print(f"  Recall   : {recall_score(y_test, baseline_pred, average='weighted'):.4f}")
print(f"  F1       : {f1_score(y_test, baseline_pred, average='weighted'):.4f}")

## 19 · LazyPredict Benchmark

**Not applicable for NLP text classification tasks.** LazyPredict is designed for tabular classification/regression and does not handle sparse TF-IDF matrices or text-specific pipelines effectively.

## 20 · FLAML AutoML Run

**Not applicable for NLP text classification tasks.** FLAML is designed for tabular AutoML. For NLP, dedicated frameworks like ModernBERT fine-tuning or Hugging Face AutoTrain are preferred.

## 21 · Additional Models

We train **Logistic Regression** and **LinearSVC** — both strong performers on TF-IDF text features.

## 22 · Top 3 Model Selection

Based on our choices, we compare: MultinomialNB, Logistic Regression, and LinearSVC.

## 23 · Final Training and Evaluation of Top 3

In [ ]:
models = {
    "MultinomialNB": MultinomialNB(alpha=1.0),
    "LogisticRegression": LogisticRegression(max_iter=1000, class_weight="balanced",
                                              random_state=SEED),
    "LinearSVC": LinearSVC(max_iter=2000, class_weight="balanced", random_state=SEED),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    preds = model.predict(X_test)
    acc = accuracy_score(y_test, preds)
    f1w = f1_score(y_test, preds, average="weighted")
    prec = precision_score(y_test, preds, average="weighted")
    rec = recall_score(y_test, preds, average="weighted")
    results[name] = {"accuracy": acc, "f1": f1w, "precision": prec, "recall": rec}
    print(f"{name}: Acc={acc:.4f} F1={f1w:.4f} Prec={prec:.4f} Rec={rec:.4f}")

res_df = pd.DataFrame(results).T.sort_values("f1", ascending=False)
print("\nRanking:")
print(res_df)

## 24 · Error Analysis

In [ ]:
best_name = res_df.index[0]
best_model = models[best_name]
preds_best = best_model.predict(X_test)

cm = confusion_matrix(y_test, preds_best)
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=["Not Hate", "Hate"])
fig, ax = plt.subplots(figsize=(6, 5))
disp.plot(ax=ax, cmap="Blues")
ax.set_title(f"Confusion Matrix — {best_name}")
plt.tight_layout()
plt.savefig(os.path.join(SAVE_DIR, "confusion_matrix.png"), dpi=100)
plt.show()

print(classification_report(y_test, preds_best, target_names=["Not Hate", "Hate"]))

# Show some misclassified examples
test_texts = X_test_text.reset_index(drop=True)
true_labels = y_test.reset_index(drop=True)
wrong_mask = preds_best != true_labels.values
wrong_idx = np.where(wrong_mask)[0]
print(f"Total errors: {len(wrong_idx)} / {len(preds_best)} ({len(wrong_idx)/len(preds_best)*100:.1f}%)")
print("\nSample misclassifications:")
for i in wrong_idx[:5]:
    print(f"  True={true_labels.iloc[i]} Pred={preds_best[i]}: {test_texts.iloc[i][:120]}...")

## 25 · Interpretation and Business Insight

- **Logistic Regression and LinearSVC** typically outperform MultinomialNB on hate speech detection because they can learn more complex decision boundaries.
- **class_weight='balanced'** is critical for imbalanced hate-speech datasets to avoid the model simply predicting the majority class.
- **Recall for hate speech (label=1)** is often the most important metric — missing actual hate speech is more costly than flagging borderline content.
- In production, these models would be combined with transformer-based models (ModernBERT) for higher accuracy.

## 26 · Limitations

- TF-IDF cannot capture semantic meaning, sarcasm, or context.
- The dataset represents a specific definition of hate speech — other taxonomies may differ.
- Single-sentence classification misses conversation context.
- A transformer-based approach would significantly outperform these baselines.

## 27 · How to Improve This Project

1. Fine-tune **ModernBERT** or **XLM-RoBERTa** for much higher performance.
2. Add data augmentation (back-translation, synonym replacement).
3. Use ensemble methods combining TF-IDF classifiers with transformer predictions.
4. Add more text features: sentiment scores, linguistic features, user metadata.
5. Perform cross-validation instead of a single train/test split.

## 28 · Production Considerations

- **Latency**: TF-IDF + LogReg is extremely fast at inference (<1ms per tweet) — ideal for real-time moderation.
- **Transformer models** give better accuracy but require GPU and have higher latency.
- **Monitoring**: Track precision/recall over time. Hate speech patterns evolve.
- **Human-in-the-loop**: Always have human review for edge cases.
- **Bias auditing**: Test for bias across different demographics and topics.

## 29 · Common Mistakes

1. Not using `class_weight='balanced'` on imbalanced hate-speech data.
2. Using accuracy alone — F1 and recall for the minority class are far more important.
3. Leaking test data into TF-IDF vocabulary by fitting the vectorizer on the entire dataset.
4. Ignoring the social and ethical implications of false positives and false negatives.
5. Deploying without regular retraining — online language evolves rapidly.

## 30 · Mini Challenge / Exercises

1. **Try character n-grams**: Change `TfidfVectorizer(analyzer='char_wb', ngram_range=(3,5))` — does it improve?
2. **Threshold tuning**: For LogisticRegression, use `predict_proba()` and find the optimal threshold for F1.
3. **Feature importance**: For LogReg, examine the top positive/negative coefficients — what words indicate hate speech?

## 31 · Final Summary / Key Takeaways

In [ ]:
metrics = {}
for name, m in results.items():
    metrics[name] = m
best = res_df.index[0]
metrics["best_model"] = best
metrics["best_accuracy"] = results[best]["accuracy"]
metrics["best_f1"] = results[best]["f1"]

with open(os.path.join(SAVE_DIR, "metrics.json"), "w") as f:
    json.dump(metrics, f, indent=2)
print("Metrics saved to metrics.json")
print(json.dumps(metrics, indent=2))

print("\n✅ Cyberbullying Classification notebook complete.")